In [ ]:
# ============================================================
# BUILD APK JAGJAG DI GOOGLE COLAB (PULL + LANJUTKAN)
# ============================================================
import os, sys, glob, shutil, hashlib, subprocess
from google.colab import files

os.chdir("/content")

REPO_URL = "https://github.com/IsekaiRid/tugas-minggu-3.git"
BRANCH   = "main"
WORKDIR  = "/content/jagjag"
LOG      = "/content/build.log"
JAVA17   = "/usr/lib/jvm/java-17-openjdk-amd64"

ENV = dict(os.environ,
           JAVA_HOME=JAVA17,
           PATH=f"{JAVA17}/bin:" + os.environ["PATH"],
           PYTHONUNBUFFERED="1")

def sh(cmd, log=None, must=True):
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    f = open(log, "a") if log else None
    for line in p.stdout:
        sys.stdout.write(line)
        if f: f.write(line)
    p.wait()
    if f: f.close()
    if must and p.returncode != 0:
        raise SystemExit(f"\nLANGKAH GAGAL (kode {p.returncode}): {cmd}")
    return p.returncode

# 1. Dependensi (dilewati kalau sudah terpasang)
if not os.path.isdir(JAVA17):
    sh("sudo apt-get update -y")
    sh("sudo apt-get install -y git zip unzip openjdk-17-jdk python3-pip autoconf "
       "libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev "
       "cmake libffi-dev libssl-dev build-essential ccache")
else:
    print("Java 17 sudah ada, instal apt dilewati.")

if not shutil.which("buildozer"):
    sh("pip install -q --upgrade pip setuptools wheel")
    sh('pip install -q buildozer "cython==0.29.36" virtualenv')
else:
    print("Buildozer sudah ada, instal pip dilewati.")

# 2. Ambil kode: PULL kalau folder ada, CLONE kalau belum
if os.path.isdir(f"{WORKDIR}/.git"):
    os.chdir(WORKDIR)
    sh("git fetch origin")
    sh(f"git reset --hard origin/{BRANCH}")     # buang perubahan lokal, ikut GitHub
    sh("git log -1 --oneline")                   # tampilkan commit yang dipakai
else:
    sh(f"rm -rf {WORKDIR}")
    sh(f"git clone {REPO_URL} {WORKDIR}")
    os.chdir(WORKDIR)

# Kalau project ada di subfolder
if not os.path.exists("buildozer.spec") and os.path.exists("JAGJAG-APP/buildozer.spec"):
    os.chdir("JAGJAG-APP")
print("Folder kerja:", os.getcwd())

# 3. Kalau pengaturan p4a/NDK/API/requirements berubah, bersihkan cache build lama
kunci = []
for baris in open("buildozer.spec"):
    b = baris.strip()
    if b.startswith(("p4a.", "android.ndk", "android.api", "requirements")):
        kunci.append(b)
sidik = hashlib.md5("\n".join(kunci).encode()).hexdigest()
marker = ".buildozer/.spec_hash"

lama = open(marker).read() if os.path.exists(marker) else None
if lama != sidik:
    print("Pengaturan build berubah: cache build lama dibersihkan (SDK/NDK tetap).")
    sh("rm -rf .buildozer/android/platform/build-* .buildozer/android/platform/python-for-android "
       ".buildozer/android/app", must=False)
    os.makedirs(".buildozer", exist_ok=True)
    open(marker, "w").write(sidik)
else:
    print("Pengaturan build sama: melanjutkan dari cache.")

# 4. Cek file penting
hilang = False
for p in ["buildozer.spec", "main.py", "assets/icon/icon.png", "assets/img/presplash.jpg"]:
    ada = os.path.exists(p)
    hilang = hilang or not ada
    print(("OK      " if ada else "HILANG  ") + p)
if hilang:
    raise SystemExit("Ada file yang hilang. Perbaiki dulu di repo.")

# 5. Build (melanjutkan dari cache yang ada)
if os.path.exists(LOG):
    os.remove(LOG)
kode = sh("yes | buildozer -v android debug", log=LOG, must=False)

# 6. Hasil
apks = glob.glob("bin/*.apk")
if apks:
    print("\nBERHASIL:", apks[0])
    files.download(apks[0])
else:
    print(f"\nBUILD GAGAL (kode {kode}). 80 baris terakhir:\n" + "=" * 60)
    with open(LOG) as f:
        print("".join(f.readlines()[-80:]))
    print("=" * 60)

Streaming output truncated to the last 5000 lines.
- Download 67.05%
- Download 67.07%
- Download 67.08%
- Download 67.10%
- Download 67.12%
- Download 67.14%
- Download 67.15%
- Download 67.17%
- Download 67.19%
- Download 67.21%
- Download 67.22%
- Download 67.24%
- Download 67.26%
- Download 67.28%
- Download 67.29%
- Download 67.31%
- Download 67.33%
- Download 67.34%
- Download 67.36%
- Download 67.38%
- Download 67.40%
- Download 67.41%
- Download 67.43%
- Download 67.45%
- Download 67.47%
- Download 67.48%
- Download 67.50%
- Download 67.52%
- Download 67.54%
- Download 67.55%
- Download 67.57%
- Download 67.59%
- Download 67.61%
- Download 67.62%
- Download 67.64%
- Download 67.66%
- Download 67.67%
- Download 67.69%
- Download 67.71%
- Download 67.73%
- Download 67.74%
- Download 67.76%
- Download 67.78%
- Download 67.80%
- Download 67.81%
- Download 67.83%
- Download 67.85%
- Download 67.87%
- Download 67.88%
- Download 67.90%
- Download 67.92%
- Download 67.94%
- Download 67

KeyboardInterrupt: 

In [ ]:
# ============================================================
# PULL + BUILD ULANG JAGJAG (jalankan setelah setup pertama)
# ============================================================
import os, sys, glob, shutil, hashlib, subprocess
from google.colab import files

os.chdir("/content")

WORKDIR = "/content/jagjag"
BRANCH  = "main"
LOG     = "/content/build.log"
JAVA17  = "/usr/lib/jvm/java-17-openjdk-amd64"

ENV = dict(os.environ,
           JAVA_HOME=JAVA17,
           PATH=f"{JAVA17}/bin:" + os.environ["PATH"],
           PYTHONUNBUFFERED="1")

def sh(cmd, log=None, must=True):
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    f = open(log, "a") if log else None
    for line in p.stdout:
        sys.stdout.write(line)
        if f: f.write(line)
    p.wait()
    if f: f.close()
    if must and p.returncode != 0:
        raise SystemExit(f"\nLANGKAH GAGAL (kode {p.returncode}): {cmd}")
    return p.returncode

# 0. Pastikan setup pertama sudah pernah dijalankan
if not (os.path.isdir(f"{WORKDIR}/.git") and shutil.which("buildozer") and os.path.isdir(JAVA17)):
    raise SystemExit("Setup belum ada (runtime mungkin direset). "
                     "Jalankan dulu script lengkap (clone + instal), lalu pakai script ini.")

# 1. Pull
os.chdir(WORKDIR)
sh("git fetch origin")
sh(f"git reset --hard origin/{BRANCH}")
sh("git log -1 --oneline")

if not os.path.exists("buildozer.spec") and os.path.exists("JAGJAG-APP/buildozer.spec"):
    os.chdir("JAGJAG-APP")

# 2. Peringatan root dimatikan di spec lokal Colab
sh(r"sed -i '/^warn_on_root/d' buildozer.spec")
sh(r"sed -i 's/^\[buildozer\]/[buildozer]\nwarn_on_root = 0/' buildozer.spec")

# 3. Bersihkan cache hanya kalau pengaturan build berubah
kunci = [b.strip() for b in open("buildozer.spec")
         if b.strip().startswith(("p4a.", "android.ndk", "android.api", "requirements"))]
sidik = hashlib.md5("\n".join(kunci).encode()).hexdigest()
marker = ".buildozer/.spec_hash"
if (open(marker).read() if os.path.exists(marker) else None) != sidik:
    print("Pengaturan build berubah: cache build dibersihkan (SDK/NDK tetap).")
    sh("rm -rf .buildozer/android/platform/build-* "
       ".buildozer/android/platform/python-for-android .buildozer/android/app", must=False)
    os.makedirs(".buildozer", exist_ok=True)
    open(marker, "w").write(sidik)
else:
    print("Pengaturan build sama: lanjut dari cache.")

# 4. Cek file penting
hilang = False
for p in ["buildozer.spec", "main.py", "assets/icon/icon.png", "assets/img/presplash.jpg"]:
    ada = os.path.exists(p)
    hilang = hilang or not ada
    print(("OK      " if ada else "HILANG  ") + p)
if hilang:
    raise SystemExit("Ada file yang hilang di repo.")

# 5. Hapus APK lama supaya yang terunduh pasti hasil build ini
for a in glob.glob("bin/*.apk"):
    os.remove(a)

# 6. Build
if os.path.exists(LOG):
    os.remove(LOG)
kode = sh("yes | buildozer -v android debug", log=LOG, must=False)

# 7. Hasil
apks = glob.glob("bin/*.apk")
if apks:
    print("\nBERHASIL:", apks[0])
    files.download(apks[0])
else:
    print(f"\nBUILD GAGAL (kode {kode}). 80 baris terakhir:\n" + "=" * 60)
    with open(LOG) as f:
        print("".join(f.readlines()[-80:]))
    print("=" * 60)

Streaming output truncated to the last 5000 lines.
Compiling '/content/jagjag/.buildozer/android/platform/build-arm64-v8a/build/other_builds/python3/arm64-v8a__ndk_target_24/python3/Lib/test/test_frame.py'...
Compiling '/content/jagjag/.buildozer/android/platform/build-arm64-v8a/build/other_builds/python3/arm64-v8a__ndk_target_24/python3/Lib/test/test_frozen.py'...
Compiling '/content/jagjag/.buildozer/android/platform/build-arm64-v8a/build/other_builds/python3/arm64-v8a__ndk_target_24/python3/Lib/test/test_fstring.py'...
Compiling '/content/jagjag/.buildozer/android/platform/build-arm64-v8a/build/other_builds/python3/arm64-v8a__ndk_target_24/python3/Lib/test/test_ftplib.py'...
Compiling '/content/jagjag/.buildozer/android/platform/build-arm64-v8a/build/other_builds/python3/arm64-v8a__ndk_target_24/python3/Lib/test/test_funcattrs.py'...
Compiling '/content/jagjag/.buildozer/android/platform/build-arm64-v8a/build/other_builds/python3/arm64-v8a__ndk_target_24/python3/Lib/test/test_functo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>